# FloodRisk Sri Lanka — Additional 33 Hydrometric Catchments (MERIT Hydro + HydroBASINS)

**Project:** J26-DS-314 | **Dataset stage:** candidate AOI delineation | **Source:** Irrigation Department ArcGIS public dashboard station GeoJSON.

We delineate **only the 33 dashboard gauges outside Kelani**. The previously generated seven Kelani catchments are neither reprocessed nor overwritten. We retain original dashboard coordinates and separately record snapped computational outlets. We use MERIT Hydro's native D8 network for delineation and HydroBASINS Level 12 for **structural cross-checks, not exact catchment equivalence**.

**Outputs:** For each successfully delineated station: one irregular GeoJSON watershed polygon, one native-grid GeoTIFF mask, one GeoJSON bounding box, plus candidate QA registry, outlet records, manifest and browser QA maps. Products are written **directly** to `shared/spatial/catchments/<basin>/`. Cache files are kept under `results/catchments/additional_33_merit_hybas12/`. There is **no Section I / second-copy publishing step**.

**Evidence policy:** Candidate does not mean verified. We flag reference gaps, implausible outlet distances/areas, coastline and flow-network issues, raster-edge truncation and source-coordinate discrepancies. We never create a new or imaginary drainage reference to make an outlet pass.

[MERIT Hydro GEE catalog](https://developers.google.com/earth-engine/datasets/catalog/MERIT_Hydro_v1_0_1) · [HydroBASINS L12](https://developers.google.com/earth-engine/datasets/catalog/WWF_HydroSHEDS_v1_Basins_hybas_12) · [ArcGIS dashboard](https://slirrigation.maps.arcgis.com/apps/dashboards/2cffe83c9ff5497d97375498bdf3ff38#)

## A. Environment, source coordinates and drainage-area reconciliation

We use the GeoJSON **geometry** `[longitude, latitude]`, not the rounded text properties. Reference drainage areas come from the Irrigation Department Hydrological Annual 2021/22 table supplied with this study.

In [ ]:
import os, re, json, math, hashlib, time, warnings, webbrowser
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED
import numpy as np, pandas as pd, requests, rasterio, folium, ee
from rasterio.windows import Window, transform as window_transform
from rasterio.transform import Affine, rowcol, xy
from rasterio.features import shapes
from shapely.geometry import shape, mapping, box
from shapely.ops import unary_union
from shapely.validation import explain_validity
from pyproj import Geod
from folium import plugins
from IPython.display import display

REPO_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
WORK_DIR = REPO_ROOT / 'results' / 'catchments' / 'additional_33_merit_hybas12'
SHARED_ROOT = REPO_ROOT / 'shared' / 'spatial' / 'catchments'
WORK_DIR.mkdir(parents=True, exist_ok=True)
SHARED_ROOT.mkdir(parents=True, exist_ok=True)
assert (SHARED_ROOT / 'kelani').resolve() != WORK_DIR.resolve()
print('Working cache:', WORK_DIR.resolve())
print('Shared outputs:', SHARED_ROOT.resolve())

EE_PROJECT = 'multimodal-flood-risk'  # Change only if the GEE project changes.
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
MERIT = ee.Image('MERIT/Hydro/v1_0_1')
HYB = ee.FeatureCollection('WWF/HydroSHEDS/v1/Basins/hybas_12')
geod = Geod(ellps='WGS84')
print('Earth Engine ready:', EE_PROJECT)

Working cache: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12
Shared outputs: C:\Users\ASUS\projects\rp_project\J26-DS-314\shared\spatial\catchments
Earth Engine ready: multimodal-flood-risk


In [36]:
# Corrected 33 non-Kelani stations, reconciled to the 40-station
# Hydrological Annual. ArcGIS point geometry coordinates are preserved.
import json
import re
import pandas as pd

ADDITIONAL_ARCGIS_GEOJSON = json.loads(r'''{"type":"FeatureCollection","features":[{"type":"Feature","id":8,"geometry":{"type":"Point","coordinates":[80.0615110003587,6.6143240002055]},"properties":{"objectid":8,"no_":8,"station":"Putupaula","latitude":6.614324,"longitude":80.061511,"basin":"Kalu Ganga","Tributory":"Kalu Ganga","Alert_Level":3,"Minor_Flood_Level":4,"Major_Flood_Level":5,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":9,"geometry":{"type":"Point","coordinates":[80.2182539999508,6.7315540003244]},"properties":{"objectid":9,"no_":9,"station":"Ellagawa","latitude":6.731554,"longitude":80.218254,"basin":"Kalu Ganga","Tributory":"Kalu Ganga","Alert_Level":10,"Minor_Flood_Level":11,"Major_Flood_Level":12,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":10,"geometry":{"type":"Point","coordinates":[80.3972350001154,6.67906700028711]},"properties":{"objectid":10,"no_":10,"station":"Rathnapura","latitude":6.679067,"longitude":80.397235,"basin":"Kalu Ganga","Tributory":"Kalu Ganga","Alert_Level":5.2,"Minor_Flood_Level":8,"Major_Flood_Level":10,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":11,"geometry":{"type":"Point","coordinates":[80.1606899996635,6.63116599994976]},"properties":{"objectid":11,"no_":11,"station":"Kalawellawa (Millakanda)","latitude":6.631166,"longitude":80.16069,"basin":"Kalu Ganga","Tributory":"Kuda Ganga","Alert_Level":5,"Minor_Flood_Level":6,"Major_Flood_Level":8,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":12,"geometry":{"type":"Point","coordinates":[80.2439349995639,6.51372700024646]},"properties":{"objectid":12,"no_":12,"station":"Magura","latitude":6.513727,"longitude":80.243935,"basin":"Kalu Ganga","Tributory":"Maguru Ganga","Alert_Level":4,"Minor_Flood_Level":6,"Major_Flood_Level":8,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":13,"geometry":{"type":"Point","coordinates":[80.1753810002093,6.17380599991107]},"properties":{"objectid":13,"no_":13,"station":"Baddegama","latitude":6.173806,"longitude":80.175381,"basin":"Gin Ganga","Tributory":"Gin Ganga","Alert_Level":3.5,"Minor_Flood_Level":4,"Major_Flood_Level":5,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":14,"geometry":{"type":"Point","coordinates":[80.3306159997305,6.34177600033244]},"properties":{"objectid":14,"no_":14,"station":"Thawalama","latitude":6.341776,"longitude":80.330616,"basin":"Gin Ganga","Tributory":"Gin Ganga","Alert_Level":4,"Minor_Flood_Level":6,"Major_Flood_Level":8,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":15,"geometry":{"type":"Point","coordinates":[80.4823839999218,6.11178000039592]},"properties":{"objectid":15,"no_":15,"station":"Panadugama","latitude":6.11178,"longitude":80.482384,"basin":"Nilwala Ganga","Tributory":"Nilwala Ganga","Alert_Level":5,"Minor_Flood_Level":6,"Major_Flood_Level":8,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":16,"geometry":{"type":"Point","coordinates":[80.4766040004319,6.21374599983565]},"properties":{"objectid":16,"no_":16,"station":"Pitabeddara","latitude":6.213746,"longitude":80.476604,"basin":"Nilwala Ganga","Tributory":"Nilwala Ganga","Alert_Level":4,"Minor_Flood_Level":5,"Major_Flood_Level":6,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":17,"geometry":{"type":"Point","coordinates":[80.5720109999098,6.23672999982568]},"properties":{"objectid":17,"no_":17,"station":"Urawa","latitude":6.23673,"longitude":80.572011,"basin":"Nilwala Ganga","Tributory":"Urubokka Ganga","Alert_Level":2.5,"Minor_Flood_Level":4,"Major_Flood_Level":6,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":18,"geometry":{"type":"Point","coordinates":[80.5329300000867,6.01029999964291]},"properties":{"objectid":18,"no_":18,"station":"Thalgahagoda","latitude":6.0103,"longitude":80.53293,"basin":"Nilwala Ganga","Tributory":"Nilwala Ganga","Alert_Level":1.4,"Minor_Flood_Level":2,"Major_Flood_Level":3,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":19,"geometry":{"type":"Point","coordinates":[80.901243999799,6.34533899959541]},"properties":{"objectid":19,"no_":19,"station":"Moraketiya","latitude":6.345339,"longitude":80.901244,"basin":"Walawe Ganga","Tributory":"Walawe Ganga","Alert_Level":3,"Minor_Flood_Level":3,"Major_Flood_Level":5,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":20,"geometry":{"type":"Point","coordinates":[81.1337510002852,6.4690499999262]},"properties":{"objectid":20,"no_":20,"station":"Thanamalwila","latitude":6.46905,"longitude":81.133751,"basin":"Kirindi Oya","Tributory":"Kirindi Oya","Alert_Level":4,"Minor_Flood_Level":5,"Major_Flood_Level":6,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":21,"geometry":{"type":"Point","coordinates":[81.1121400004247,6.69575999959604]},"properties":{"objectid":21,"no_":21,"station":"Wellawaya","latitude":6.69576,"longitude":81.11214,"basin":"Kirindi Oya","Tributory":"Kirindi Oya","Alert_Level":4.4,"Minor_Flood_Level":5,"Major_Flood_Level":6,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":22,"geometry":{"type":"Point","coordinates":[81.1233370000797,6.52474599995434]},"properties":{"objectid":22,"no_":22,"station":"Kuda Oya","latitude":6.524746,"longitude":81.123337,"basin":"Kirindi Oya","Tributory":"Kuda Oya","Alert_Level":6.9,"Minor_Flood_Level":8,"Major_Flood_Level":9,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":23,"geometry":{"type":"Point","coordinates":[81.3321899999841,6.417705000347]},"properties":{"objectid":23,"no_":23,"station":"Katharagama","latitude":6.417705,"longitude":81.33219,"basin":"Menik Ganga","Tributory":"Menik Ganga","Alert_Level":4,"Minor_Flood_Level":5,"Major_Flood_Level":6,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":24,"geometry":{"type":"Point","coordinates":[81.2970650001108,6.8953620001963]},"properties":{"objectid":24,"no_":24,"station":"Nakkala","latitude":6.895362,"longitude":81.297065,"basin":"Kumbukkan Oya","Tributory":"Kumbukkan Oya","Alert_Level":5,"Minor_Flood_Level":6,"Major_Flood_Level":8,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":25,"geometry":{"type":"Point","coordinates":[81.545409999742,6.90591099956999]},"properties":{"objectid":25,"no_":25,"station":"Siyambalanduwa","latitude":6.905911,"longitude":81.54541,"basin":"Heda Oya","Tributory":"Heda Oya","Alert_Level":4.5,"Minor_Flood_Level":6,"Major_Flood_Level":7,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":27,"geometry":{"type":"Point","coordinates":[81.1946370001375,7.38225100016999]},"properties":{"objectid":27,"no_":27,"station":"Padiyathalawa","latitude":7.382251,"longitude":81.194637,"basin":"Maduru Oya","Tributory":"Maduru Oya","Alert_Level":4,"Minor_Flood_Level":4,"Major_Flood_Level":6,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":28,"geometry":{"type":"Point","coordinates":[81.0905939999633,7.91298699959734]},"properties":{"objectid":28,"no_":28,"station":"Manampitiya","latitude":7.912987,"longitude":81.090594,"basin":"Mahaweli Ganga             ","Tributory":"Mahaweli Ganga","Alert_Level":3,"Minor_Flood_Level":6,"Major_Flood_Level":7,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":29,"geometry":{"type":"Point","coordinates":[80.9891099999962,7.31646999997607]},"properties":{"objectid":29,"no_":29,"station":"Weraganthota","latitude":7.31647,"longitude":80.98911,"basin":"Mahaweli Ganga ","Tributory":"Mahaweli Ganga","Alert_Level":5,"Minor_Flood_Level":6,"Major_Flood_Level":8,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":30,"geometry":{"type":"Point","coordinates":[80.5930050000022,7.26905900032262]},"properties":{"objectid":30,"no_":30,"station":"Peradeniya","latitude":7.269059,"longitude":80.593005,"basin":"Mahaweli Ganga             ","Tributory":"Mahaweli Ganga","Alert_Level":5,"Minor_Flood_Level":7,"Major_Flood_Level":9,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":31,"geometry":{"type":"Point","coordinates":[80.53616208997,7.04894547008318]},"properties":{"objectid":31,"no_":31,"station":"Nawalapitiya","latitude":7.04894547,"longitude":80.53616209,"basin":"Mahaweli Ganga","Tributory":"Mahaweli Ganga","Alert_Level":3.5,"Minor_Flood_Level":5,"Major_Flood_Level":6,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":32,"geometry":{"type":"Point","coordinates":[81.048886000161,7.09065299961578]},"properties":{"objectid":32,"no_":32,"station":"Thaldena","latitude":7.090653,"longitude":81.048886,"basin":"Mahaweli Ganga","Tributory":"Badulu Oya","Alert_Level":3,"Minor_Flood_Level":4,"Major_Flood_Level":5,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":34,"geometry":{"type":"Point","coordinates":[80.8808149995961,8.57699799998582]},"properties":{"objectid":34,"no_":34,"station":"Horowpothana","latitude":8.576998,"longitude":80.880815,"basin":"Yan Oya                          ","Tributory":"Yan Oya","Alert_Level":6,"Minor_Flood_Level":8,"Major_Flood_Level":10,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":35,"geometry":{"type":"Point","coordinates":[80.6749679997877,8.72603100017587]},"properties":{"objectid":35,"no_":35,"station":"Yaka Wewa","latitude":8.726031,"longitude":80.674968,"basin":"Ma Oya","Tributory":"Mukunu Oya","Alert_Level":4,"Minor_Flood_Level":5,"Major_Flood_Level":6,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":36,"geometry":{"type":"Point","coordinates":[80.2780700002985,8.59008099995541]},"properties":{"objectid":36,"no_":36,"station":"Thanthirimale","latitude":8.590081,"longitude":80.27807,"basin":"Malwathu Oya","Tributory":"Malwathu Oya","Alert_Level":5,"Minor_Flood_Level":7,"Major_Flood_Level":8,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":37,"geometry":{"type":"Point","coordinates":[80.2623219997686,7.96936799958593]},"properties":{"objectid":37,"no_":37,"station":"Galgamuwa","latitude":7.969368,"longitude":80.262322,"basin":"Mee Oya","Tributory":"Mee Oya","Alert_Level":4.84,"Minor_Flood_Level":6,"Major_Flood_Level":8,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":38,"geometry":{"type":"Point","coordinates":[79.99659,7.70003500000002]},"properties":{"objectid":38,"no_":38,"station":"Moragaswewa","latitude":7.7,"longitude":79.99,"basin":"Deduru Oya","Tributory":"Deduru Oya","Alert_Level":4.75,"Minor_Flood_Level":null,"Major_Flood_Level":null,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":39,"geometry":{"type":"Point","coordinates":[79.9839160002518,7.30074100013236]},"properties":{"objectid":39,"no_":39,"station":"Badalgama","latitude":7.300741,"longitude":79.983916,"basin":"Maha Oya","Tributory":"Maha Oya","Alert_Level":5,"Minor_Flood_Level":6,"Major_Flood_Level":10,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":40,"geometry":{"type":"Point","coordinates":[80.1250950003353,7.33073600009912]},"properties":{"objectid":40,"no_":40,"station":"Giriulla","latitude":7.330736,"longitude":80.125095,"basin":"Maha Oya","Tributory":"Maha Oya","Alert_Level":5.5,"Minor_Flood_Level":6,"Major_Flood_Level":8,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":41,"geometry":{"type":"Point","coordinates":[80.0840849995936,7.11641600021263]},"properties":{"objectid":41,"no_":41,"station":"Dunamale","latitude":7.116416,"longitude":80.084085,"basin":"Aththanagalu Oya","Tributory":"Attanagalu Oya","Alert_Level":3.3,"Minor_Flood_Level":4,"Major_Flood_Level":6,"Unit":"m","Elivation_m_MSL":null}},{"type":"Feature","id":42,"geometry":{"type":"Point","coordinates":[80.3255579995907,8.12126900009338]},"properties":{"objectid":42,"no_":42,"station":"Thambuththegama","latitude":8.121269,"longitude":80.325558,"basin":"Kala Oya","Tributory":"Kala Oya","Alert_Level":4,"Minor_Flood_Level":5,"Major_Flood_Level":6,"Unit":"m","Elivation_m_MSL":null}}]}''')

PUBLISHED_DRAINAGE_KM2 = {'Rathnapura': 603, 'Ellagawa': 1393, 'Kalawellawa (Millakanda)': 780, 'Magura': 152, 'Putupaula': 2598, 'Baddegama': 749, 'Thawalama': 377, 'Urawa': 59, 'Pitabeddara': 310, 'Panadugama': 445, 'Thalgahagoda': 852, 'Moraketiya': 1542, 'Wellawaya': 172, 'Thanamalwila': 749, 'Kuda Oya': 291, 'Katharagama': 787, 'Nakkala': 216, 'Siyambalanduwa': 295, 'Padiyathalawa': 159, 'Thaldena': 276, 'Nawalapitiya': 176, 'Peradeniya': 1168, 'Weraganthota': 4092, 'Manampitiya': 7418, 'Horowpothana': 720, 'Yaka Wewa': 121, 'Thanthirimale': 2116, 'Galgamuwa': 299, 'Moragaswewa': 1708, 'Giriulla': 1191, 'Badalgama': 1360, 'Dunamale': 153, 'Thambuththegama': 1186}

KELANI_STATIONS = {
    'Nagalagam Street', 'Hanwella', 'Glencourse', 'Kithulgala',
    'Holombuwa', 'Deraniyagala', 'Norwood'
}

def slugify(s):
    return re.sub(r'[^a-z0-9]+', '_', str(s).lower()).strip('_')

rows = []
for f in ADDITIONAL_ARCGIS_GEOJSON['features']:
    p = f['properties']
    lon, lat = map(float, f['geometry']['coordinates'])
    name = p['station'].strip()
    assert name not in KELANI_STATIONS
    assert name in PUBLISHED_DRAINAGE_KM2, f'Missing area reference: {name}'
    notes = []
    if (abs(lon - float(p.get('longitude', lon))) > 0.001 or
            abs(lat - float(p.get('latitude', lat))) > 0.001):
        notes.append('GEOMETRY_VS_PROPERTY_COORDINATES_DIFFER')
    rows.append({
        'station': name, 'basin': p['basin'].strip(),
        'tributary': p.get('Tributory', '').strip(),
        'lon': lon, 'lat': lat,
        'official_area_km2': PUBLISHED_DRAINAGE_KM2[name],
        'source_objectid': f.get('id'),
        'reference_notes': ';'.join(notes)
    })

GAUGES = pd.DataFrame(rows)
assert len(GAUGES) == 33 and GAUGES.station.nunique() == 33
assert GAUGES.lon.between(79, 82.2).all() and GAUGES.lat.between(5.8, 9).all()
assert GAUGES.official_area_km2.notna().sum() == 33
assert GAUGES.basin.nunique() == 18
assert not {'Ampara','Calidonia'}.intersection(GAUGES.station)
assert {'Thambuththegama','Moragaswewa'}.issubset(set(GAUGES.station))
assert set(GAUGES.station) == set(PUBLISHED_DRAINAGE_KM2)

GAUGES.to_csv(WORK_DIR / 'additional_33_source_gauge_registry.csv', index=False)
print('Gauges:', len(GAUGES), '| Basin groups:', GAUGES.basin.nunique(),
      '| With published area:', int(GAUGES.official_area_km2.notna().sum()))
display(GAUGES[['station', 'basin', 'tributary', 'lon', 'lat',
                'official_area_km2', 'reference_notes']])


Gauges: 33 | Basin groups: 18 | With published area: 33


,station,basin,tributary,lon,lat,official_area_km2,reference_notes
0,Putupaula,Kalu Ganga,Kalu Ganga,80.061511,6.614324,2598,
1,Ellagawa,Kalu Ganga,Kalu Ganga,80.218254,6.731554,1393,
2,Rathnapura,Kalu Ganga,Kalu Ganga,80.397235,6.679067,603,
3,Kalawellawa (Millakanda),Kalu Ganga,Kuda Ganga,80.160690,6.631166,780,
4,Magura,Kalu Ganga,Maguru Ganga,80.243935,6.513727,152,
5,Baddegama,Gin Ganga,Gin Ganga,80.175381,6.173806,749,
6,Thawalama,Gin Ganga,Gin Ganga,80.330616,6.341776,377,
7,Panadugama,Nilwala Ganga,Nilwala Ganga,80.482384,6.111780,445,
8,Pitabeddara,Nilwala Ganga,Nilwala Ganga,80.476604,6.213746,310,
9,Urawa,Nilwala Ganga,Urubokka Ganga,80.572011,6.236730,59,


## B. Basemap and optional nationwide reference map

We use **Esri Topographic** and **Esri Satellite**, as in the Kelani workflow. CARTO Voyager is optional if our separate local secret is configured, we never embed credentials into source code. The HTML map is saved to disk, avoiding VS Code Jupyter trust problems. **The 18-basin nationwide compatibility screen has already been performed in the Kelani pilot, it is not rerun here.**

In [37]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENABLE_CARTO_VOYAGER = True

ENV_FILE = Path("secrets.env").resolve()
loaded = load_dotenv(ENV_FILE)
CARTO_API_KEY = os.getenv("FLOODRISK_CARTO_BASEMAP_KEY", "").strip()

print("Environment file exists:", ENV_FILE.is_file(), flush=True)
print("Environment file loaded:", loaded, flush=True)
print("CARTO key available:", bool(CARTO_API_KEY), flush=True)

if ENABLE_CARTO_VOYAGER and not CARTO_API_KEY:
    raise RuntimeError(
        "CARTO key missing. Check your .env file or set "
        "ENABLE_CARTO_VOYAGER=False."
    )

ESRI_URL = (
    "https://server.arcgisonline.com/ArcGIS/rest/services/"
    "World_Imagery/MapServer/tile/{z}/{y}/{x}"
)

ESRI_TOPO_URL = (
    "https://server.arcgisonline.com/ArcGIS/rest/services/"
    "World_Topo_Map/MapServer/tile/{z}/{y}/{x}"
)

ESRI_ATTRIBUTION = (
    "Tiles &copy; Esri, Maxar, Earthstar Geographics, "
    "and GIS User Community"
)

CARTO_ATTRIBUTION = (
    '&copy; <a href="https://www.openstreetmap.org/copyright">'
    'OpenStreetMap</a> contributors, '
    '&copy; <a href="https://carto.com/attribution/">CARTO</a>'
)

def make_base_map(center, zoom=8, satellite_first=False, height=620):
    m = folium.Map(
        location=center,
        zoom_start=zoom,
        tiles=None,
        control_scale=True,
        height=height
    )

    folium.TileLayer(
        tiles=ESRI_TOPO_URL,
        name="Esri Topographic",
        attr="Tiles &copy; Esri and its data providers",
        max_zoom=19,
        show=(not satellite_first and not ENABLE_CARTO_VOYAGER),
        overlay=False,
        control=True
    ).add_to(m)

    if ENABLE_CARTO_VOYAGER:
        folium.TileLayer(
            tiles=(
                "https://basemaps.cartocdn.com/rastertiles/voyager/"
                "{z}/{x}/{y}.png?key=" + CARTO_API_KEY
            ),
            name="CARTO Voyager (OSM-derived)",
            attr=CARTO_ATTRIBUTION,
            max_zoom=20,
            show=not satellite_first,
            overlay=False,
            control=True
        ).add_to(m)

    folium.TileLayer(
        tiles=ESRI_URL,
        name="Esri Satellite",
        attr=ESRI_ATTRIBUTION,
        max_zoom=19,
        show=satellite_first,
        overlay=False,
        control=True
    ).add_to(m)

    return m

def ee_tile_layer(image,vis,name,shown=True,opacity=.80):
    tile=ee.Image(image).getMapId(vis)
    return folium.TileLayer(tiles=tile['tile_fetcher'].url_format,attr='Google Earth Engine',
                            name=name,show=shown,opacity=opacity,overlay=True,control=True)

def save_browser_map(folium_map, path, open_tab=True):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    folium_map.save(str(path.resolve()))
    print('Map saved:',path.resolve())
    if open_tab:
        webbrowser.open_new_tab(path.resolve().as_uri())
    return path
print('Basemaps ready (no HTML created yet).')

Environment file exists: True
Environment file loaded: True
CARTO key available: True
Basemaps ready (no HTML created yet).


## C. Extract the **native MERIT grid once** for all 33 gauges

We export **one regional Sri Lanka-wide envelope** rather than 18 independently clipped river fragments. This avoids discontinuities where a station's upstream catchment crosses a manually selected region boundary and is simpler to reproduce. The rectangular envelope is *processing context*, not the catchment polygon. Its bounds include the whole island with a surrounding margin; **every watershed is still tested for touching the raster edge**.

All MERIT `dir` and `upa` pixels are downloaded on the **original MERIT grid**, in small verified tiles. The fully merged GeoTIFF and individual tile downloads are cached under `results/`, not GitHub. A rerun reuses existing verified outputs, no download is repeated unless files are missing/invalid. **Source MERIT pixels are not interpolated.** A failed or truncated download must be corrected, not silently used.

This step can take time on the free Earth Engine tier (roughly several dozen small GEE requests). Run it once and keep the cache.

In [4]:
# West, south, east, north. This deliberately contains the island and a wide margin.
# This rectangle is not a basin definition, and is not cropped to the gauge clusters.
EXPORT_BOUNDS = (79.32, 5.62, 82.20, 10.03)
TILE_SIZE = 640
RIVER_MIN_UPA_KM2 = 3
SEARCH_RADIUS_M = 350
FALLBACK_RADIUS_M = 900
OUTLET_PIXEL_OVERRIDES = {}  # e.g. {'Baddegama': (native_row, native_col)} after visual review.

proj=MERIT.select('dir').projection().getInfo()
src_crs=proj['crs'];src_affine=Affine(*proj['transform'])
assert src_crs=='EPSG:4326' and src_affine.a>0 and src_affine.e<0
assert abs(src_affine.b)<1e-12 and abs(src_affine.d)<1e-12
west,south,east,north=EXPORT_BOUNDS
col0=int(math.floor((west-src_affine.c)/src_affine.a))
col1=int(math.ceil((east-src_affine.c)/src_affine.a))
row0=int(math.floor((north-src_affine.f)/src_affine.e))
row1=int(math.ceil((south-src_affine.f)/src_affine.e))
H,W=row1-row0,col1-col0
local_affine=src_affine*Affine.translation(col0,row0)
assert 0<W<8000 and 0<H<8000
assert all(west<=row.lon<=east and south<=row.lat<=north for row in GAUGES.itertuples(index=False))
print(f'MERIT export: {W} x {H} native cells; uncompressed 2-band float32 ≈ {W*H*8/1024**2:.1f} MiB')

merged_path=WORK_DIR/'merit_additional33_native_dir_upa.tif'
partial_path=WORK_DIR/'merit_additional33_native_dir_upa_INCOMPLETE.tif'
profile=dict(driver='GTiff',height=H,width=W,count=2,dtype='float32',crs=src_crs,
             transform=local_affine,nodata=-9999,compress='lzw',tiled=True,
             blockxsize=256,blockysize=256)

def validate_native_raster(path,expected_h,expected_w,expected_tf):
    try:
        with rasterio.open(path) as src:
            return (src.count==2 and src.height==expected_h and src.width==expected_w and
                    src.crs.to_string()==src_crs and np.allclose(tuple(src.transform)[:6],
                    tuple(expected_tf)[:6],atol=1e-9,rtol=0))
    except Exception:
        return False

if merged_path.exists() and not validate_native_raster(merged_path,H,W,local_affine):
    raise RuntimeError('Cached merged raster has incorrect grid. Review/remove it manually, then rerun.')
if not merged_path.exists():
    merit_two=MERIT.select(['dir','upa']).toFloat().unmask(-9999)
    with rasterio.open(partial_path,'w',**profile) as dst:
        dst.set_band_description(1,'MERIT native D8 direction (ESRI pointer codes)')
        dst.set_band_description(2,'MERIT upstream area km²')
        for r0 in range(0,H,TILE_SIZE):
            for c0 in range(0,W,TILE_SIZE):
                th,tw=min(TILE_SIZE,H-r0),min(TILE_SIZE,W-c0)
                tf=window_transform(Window(c0,r0,tw,th),local_affine)
                tilepath=WORK_DIR/f'_native_tile_r{r0}_c{c0}.tif'
                if not validate_native_raster(tilepath,th,tw,tf):
                    if tilepath.exists():
                        raise RuntimeError(f'Cached tile invalid: {tilepath}; inspect it before retrying.')
                    params={'crs':src_crs,'crs_transform':list(tuple(tf)[:6]),
                            'dimensions':[tw,th],'format':'GEO_TIFF'}
                    url=merit_two.getDownloadURL(params)
                    resp=requests.get(url,timeout=180);resp.raise_for_status()
                    tmp=tilepath.with_suffix('.partial');tmp.write_bytes(resp.content)
                    if not validate_native_raster(tmp,th,tw,tf):
                        raise RuntimeError(f'Tile download failed native-grid check: {tmp}')
                    tmp.replace(tilepath)
                with rasterio.open(tilepath) as tile:
                    dst.write(tile.read(),window=Window(c0,r0,tw,th))
                print(f'Native MERIT tile r={r0} c={c0} verified',flush=True)
    partial_path.replace(merged_path)
    print('Wrote complete MERIT grid:',merged_path)
else:
    print('Reusing existing MERIT raster:',merged_path)

with rasterio.open(merged_path) as src:
    direction=src.read(1);upstream_area=src.read(2)
found=set(np.unique(direction[np.isfinite(direction)]).astype(int))
valid_dir={-9999,-1,0,1,2,4,8,16,32,64,128}
assert found.issubset(valid_dir),found-valid_dir
assert np.allclose(direction,np.round(direction),atol=1e-5)
dir_grid=direction.astype(np.int16)
print('Native MERIT grid ready:',sorted(found))

MERIT export: 3457 x 5293 native cells; uncompressed 2-band float32 ≈ 139.6 MiB
Native MERIT tile r=0 c=0 verified
Native MERIT tile r=0 c=640 verified
Native MERIT tile r=0 c=1280 verified
Native MERIT tile r=0 c=1920 verified
Native MERIT tile r=0 c=2560 verified
Native MERIT tile r=0 c=3200 verified
Native MERIT tile r=640 c=0 verified
Native MERIT tile r=640 c=640 verified
Native MERIT tile r=640 c=1280 verified
Native MERIT tile r=640 c=1920 verified
Native MERIT tile r=640 c=2560 verified
Native MERIT tile r=640 c=3200 verified
Native MERIT tile r=1280 c=0 verified
Native MERIT tile r=1280 c=640 verified
Native MERIT tile r=1280 c=1280 verified
Native MERIT tile r=1280 c=1920 verified
Native MERIT tile r=1280 c=2560 verified
Native MERIT tile r=1280 c=3200 verified
Native MERIT tile r=1920 c=0 verified
Native MERIT tile r=1920 c=640 verified
Native MERIT tile r=1920 c=1280 verified
Native MERIT tile r=1920 c=1920 verified
Native MERIT tile r=1920 c=2560 verified
Native MERIT tile

## D. Rank outlet candidates without changing original coordinates

As with Kelani, we consider cells on plausible MERIT channels near each dashboard point, ranked by distance and (where available) agreement with published drainage area. **The ranking is not independent evidence of catchment correctness.** We retain a table of alternative candidates, original dashboard locations and selected computational outlets.

Station-specific failures are recorded instead of aborting all 33 catchments. Outlet/tributary maps should be checked before treating any geometry as accepted.

In [5]:
def candidates_near(st,radius_m):
    rr,cc=rowcol(local_affine,float(st.lon),float(st.lat));rr=int(rr);cc=int(cc)
    dy=max(4,int(math.ceil(radius_m/90))+2)
    found=[]
    for r in range(max(0,rr-dy),min(H,rr+dy+1)):
        for c in range(max(0,cc-dy),min(W,cc+dy+1)):
            direction_value=int(dir_grid[r,c]);upa=float(upstream_area[r,c])
            if direction_value not in (0,1,2,4,8,16,32,64,128) or not np.isfinite(upa) or upa<RIVER_MIN_UPA_KM2:
                continue
            lon,lat=xy(local_affine,r,c,offset='center')
            distance=geod.inv(float(st.lon),float(st.lat),lon,lat)[2]
            if distance>radius_m:continue
            reference=float(st.official_area_km2)
            has_ref=np.isfinite(reference) and reference>0
            area_err=abs(math.log(max(upa,.001)/reference)) if has_ref else float('nan')
            score=area_err*2+distance/450 if has_ref else distance/450
            found.append(dict(station=st.station,basin=st.basin,grid_row=r,grid_col=c,
                              snapped_lon=float(lon),snapped_lat=float(lat),distance_m=round(distance,2),
                              merit_upa_km2=upa,official_area_km2=reference,
                              area_difference_pct=100*abs(upa-reference)/reference if has_ref else float('nan'),
                              direction=direction_value,rank_score=score))
    return sorted(found,key=lambda z:z['rank_score'])

all_candidates=[];selected=[];unresolved=[]
for st in GAUGES.itertuples(index=False):
    candidates=candidates_near(st,SEARCH_RADIUS_M)
    used_radius=SEARCH_RADIUS_M
    if not candidates:
        candidates=candidates_near(st,FALLBACK_RADIUS_M)
        used_radius=FALLBACK_RADIUS_M
    for i,rec in enumerate(candidates[:12]):
        all_candidates.append(dict(rec,ranking=i+1,used_radius_m=used_radius))
    if not candidates:
        unresolved.append(dict(station=st.station,basin=st.basin,
                               reason='NO_MERIT_RIVER_CANDIDATE_WITHIN_900M'))
        print(st.station,'UNRESOLVED — no plausible channel within 900 m')
        continue
    if st.station in OUTLET_PIXEL_OVERRIDES:
        pixel=tuple(OUTLET_PIXEL_OVERRIDES[st.station])
        picked=next((v for v in candidates if (v['grid_row'],v['grid_col'])==pixel),None)
        if picked is None:
            raise ValueError(f'Override {st.station} is not a candidate in the searched radius: {pixel}')
        method='MANUAL_PIXEL_OVERRIDE_REVIEW_REQUIRED'
    else:
        picked=candidates[0];method='AUTO_RANKED_CANDIDATE_NOT_VERIFIED'
    note=st.reference_notes
    if not np.isfinite(float(st.official_area_km2)):
        method='REVIEW_NO_OFFICIAL_DRAINAGE_REFERENCE'
    elif float(picked['area_difference_pct'])>20:
        method='REVIEW_OUTLET_AREA_DIFFERENCE_GT20'
    if used_radius>SEARCH_RADIUS_M:
        method+=';FALLBACK_RADIUS_USED'
    if note:
        method+=';'+note
    selected.append(dict(picked,original_lon=float(st.lon),original_lat=float(st.lat),
                         tributary=st.tributary,selection_method=method,
                         used_radius_m=used_radius,reference_notes=note))

candidates_df=pd.DataFrame(all_candidates)
outlets_df=pd.DataFrame(selected)
unresolved_df=pd.DataFrame(unresolved,columns=['station','basin','reason'])
candidates_df.to_csv(WORK_DIR/'outlet_candidate_rankings.csv',index=False)
outlets_df.to_csv(WORK_DIR/'selected_computational_outlets_CANDIDATE.csv',index=False)
unresolved_df.to_csv(WORK_DIR/'unresolved_outlets.csv',index=False)
print('Selected:',len(outlets_df),'of 33 | Unresolved:',len(unresolved_df))
if not outlets_df.empty:
    display(outlets_df[['station','basin','distance_m','merit_upa_km2',
                        'official_area_km2','area_difference_pct','selection_method']].round(2))
if not unresolved_df.empty:display(unresolved_df)

Selected: 33 of 33 | Unresolved: 0


,station,basin,distance_m,merit_upa_km2,official_area_km2,area_difference_pct,selection_method
0,Putupaula,Kalu Ganga,76.71,2616.68,2598.0,0.72,AUTO_RANKED_CANDIDATE_NOT_VERIFIED
1,Ellagawa,Kalu Ganga,80.18,1398.97,1393.0,0.43,AUTO_RANKED_CANDIDATE_NOT_VERIFIED
2,Rathnapura,Kalu Ganga,31.31,633.63,603.0,5.08,AUTO_RANKED_CANDIDATE_NOT_VERIFIED
3,Kalawellawa (Millakanda),Kalu Ganga,57.59,772.80,780.0,0.92,AUTO_RANKED_CANDIDATE_NOT_VERIFIED
4,Magura,Kalu Ganga,50.52,160.90,152.0,5.86,AUTO_RANKED_CANDIDATE_NOT_VERIFIED
5,Baddegama,Gin Ganga,58.04,721.52,749.0,3.67,AUTO_RANKED_CANDIDATE_NOT_VERIFIED
6,Thawalama,Gin Ganga,69.22,358.49,377.0,4.91,AUTO_RANKED_CANDIDATE_NOT_VERIFIED
7,Panadugama,Nilwala Ganga,17.94,434.15,445.0,2.44,AUTO_RANKED_CANDIDATE_NOT_VERIFIED
8,Pitabeddara,Nilwala Ganga,47.03,288.12,310.0,7.06,AUTO_RANKED_CANDIDATE_NOT_VERIFIED
9,Urawa,Nilwala Ganga,54.57,75.98,59.0,28.79,REVIEW_OUTLET_AREA_DIFFERENCE_GT20


## E. Computational outlet map (one interactive overview, saved HTML)

Red markers = immutable ArcGIS dashboard positions, yellow markers = MERIT computational outlet candidates. The connector line records the adjustment. We inspect river identity, correct tributary, confluence location and distance. Use the Esri Satellite base or CARTO if locally enabled. The map does not edit any coordinates.

In [38]:
outlet_map=make_base_map([7.35,80.75],zoom=8,satellite_first=True,height=710)
ee_tile_layer(MERIT.select('upa').gte(5).selfMask(),{'palette':['42c3e6']},
              'MERIT drainage network ≥5 km²',True,.78).add_to(outlet_map)
hyb_lines=HYB.filterBounds(ee.Geometry.Rectangle(EXPORT_BOUNDS,geodesic=False)).style(
    color='b26acb',width=1,fillColor='00000000')
ee_tile_layer(hyb_lines,{},'HydroBASINS L12 boundaries',False,.55).add_to(outlet_map)
fg_orig=folium.FeatureGroup(name='Original ArcGIS station points (red)',show=True)
fg_out=folium.FeatureGroup(name='Candidate MERIT outlets (yellow)',show=True)
for st in outlets_df.itertuples(index=False):
    folium.CircleMarker([st.original_lat,st.original_lon],radius=4,color='red',fill=True,
        tooltip=st.station+' | '+st.basin+' | ArcGIS original').add_to(fg_orig)
    folium.CircleMarker([st.snapped_lat,st.snapped_lon],radius=4,color='yellow',fill=True,
        tooltip=st.station+' | candidate MERIT outlet',
        popup=f'{st.station} | area {st.merit_upa_km2:.1f} km² | offset {st.distance_m:.0f} m').add_to(fg_out)
    folium.PolyLine([[st.original_lat,st.original_lon],[st.snapped_lat,st.snapped_lon]],
        color='orange',weight=2).add_to(fg_out)
fg_orig.add_to(outlet_map);fg_out.add_to(outlet_map)
plugins.Fullscreen().add_to(outlet_map)
folium.LayerControl(collapsed=False).add_to(outlet_map)
save_browser_map(outlet_map,WORK_DIR/'additional_33_candidate_outlets_map.html')

Map saved: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\additional_33_candidate_outlets_map.html


WindowsPath('c:/Users/ASUS/projects/rp_project/J26-DS-314/results/catchments/additional_33_merit_hybas12/additional_33_candidate_outlets_map.html')

## F. Trace the upstream D8 catchments and **write final candidate outputs directly**

We reverse-trace each outlet along MERIT's ESRI D8 directions using the same validated algorithm as in the seven-gauge pilot. We calculate the geodesic polygon area, compare it with MERIT's `upa` and the published station area, and flag outlet/search uncertainty and any raster-edge truncation. **Every file remains `CANDIDATE`, even if automatic tests pass.**

Each successful station is written **once** to `shared/spatial/catchments/<basin_slug>/{polygons,masks,bounding_boxes}/`. We do not produce or copy any Kelani catchment output, and rerunning this cell overwrites the same deterministic filenames (no duplicate versions). Existing identical outputs remain unchanged. A failed station creates a QA failure record, not a fabricated polygon.

**Hard stop:** Any watershed reaching the MERIT export edge is recorded as failed, a truncated polygon is not written to the shared directory. The other stations continue.

In [7]:
UPSTREAM_NEIGHBORS=[(0,-1,1),(-1,-1,2),(-1,0,4),(-1,1,8),
                    (0,1,16),(1,1,32),(1,0,64),(1,-1,128)]

def trace_upstream_d8(flow,seed_r,seed_c):
    nrows,ncols=flow.shape
    if not (0<=seed_r<nrows and 0<=seed_c<ncols):raise ValueError('Outlet out of source raster')
    if int(flow[seed_r,seed_c]) not in (-1,0,1,2,4,8,16,32,64,128):
        raise ValueError('Invalid native D8 outlet')
    mask=np.zeros(flow.shape,dtype=np.uint8)
    q=deque([(int(seed_r),int(seed_c))]);mask[seed_r,seed_c]=1
    while q:
        r,c=q.popleft()
        for dr,dc,ptr in UPSTREAM_NEIGHBORS:
            nr,nc=r+dr,c+dc
            if 0<=nr<nrows and 0<=nc<ncols and mask[nr,nc]==0 and int(flow[nr,nc])==ptr:
                mask[nr,nc]=1;q.append((nr,nc))
    return mask

# Preserve Kelani notebook's tested D8 convention; independent synthetic test.
_test=np.zeros((6,6),dtype=np.int16)
_test[1,1]=1;_test[1,2]=4;_test[2,2]=4;_test[3,2]=0
assert trace_upstream_d8(_test,3,2).sum()==4
print('PASS: D8 reverse-tracing synthetic test')

def write_json_if_changed(path, obj):
    path=Path(path)
    content=json.dumps(obj,ensure_ascii=False,separators=(',',':'))
    if path.exists() and path.read_text(encoding='utf8')==content:
        return
    path.write_text(content,encoding='utf8')

summary=[];combined_features=[];trace_failures=[]
for st in outlets_df.itertuples(index=False):
    station,basin=st.station,st.basin
    slug=slugify(station); basin_slug=slugify(basin)
    if basin_slug=='kelani_ganga':
        raise RuntimeError('Refusing to write to the existing Kelani catchment directory.')
    output_dir=SHARED_ROOT/basin_slug
    for sub in ['polygons','masks','bounding_boxes','metadata']:
        (output_dir/sub).mkdir(parents=True,exist_ok=True)
    try:
        watershed=trace_upstream_d8(dir_grid,int(st.grid_row),int(st.grid_col))
        touches_edge=bool(watershed[0,:].any() or watershed[-1,:].any() or
                          watershed[:,0].any() or watershed[:,-1].any())
        if touches_edge:
            raise RuntimeError('INCOMPLETE_TOUCHES_EXPORT_EDGE — enlarge native MERIT region before delineating')
        points=np.argwhere(watershed==1)
        if len(points)==0:raise ValueError('Empty watershed')
        (minr,minc),(maxr,maxc)=points.min(axis=0),points.max(axis=0)
        inner=watershed[minr:maxr+1,minc:maxc+1]
        inner_tf=window_transform(Window(int(minc),int(minr),inner.shape[1],inner.shape[0]),local_affine)
        pieces=[shape(g) for g,v in shapes(inner,mask=inner.astype(bool),transform=inner_tf) if int(v)==1]
        geom=unary_union(pieces)
        if geom.is_empty or not geom.is_valid:
            raise ValueError('Invalid catchment geometry: '+explain_validity(geom))
        area=abs(geod.geometry_area_perimeter(geom)[0])/1e6
        ref=float(st.official_area_km2);ref_ok=np.isfinite(ref) and ref>0
        pct=100*abs(area-ref)/ref if ref_ok else float('nan')
        merit_pct=100*abs(area-float(st.merit_upa_km2))/max(float(st.merit_upa_km2),.001)
        if touches_edge:
            status='INCOMPLETE_TOUCHES_EXPORT_EDGE'
        elif merit_pct>15:
            status='REVIEW_D8_VS_MERIT_UPA_GT15'
        elif not ref_ok:
            status='REVIEW_NO_OFFICIAL_DRAINAGE_REFERENCE'
        elif pct>20:
            status='REVIEW_OFFICIAL_DRAINAGE_DIFFERENCE_GT20'
        elif 'GEOMETRY_VS_PROPERTY' in st.reference_notes:
            status='REVIEW_SOURCE_COORDINATE_DISCREPANCY'
        elif st.distance_m>SEARCH_RADIUS_M:
            status='REVIEW_DISTANT_OUTLET'
        else:
            status='CANDIDATE_PENDING_VISUAL_QA'
        # Reruns replace the same deterministic files rather than generating duplicates.
        mask_file=output_dir/'masks'/f'{slug}_catchment_mask_CANDIDATE.tif'
        poly_file=output_dir/'polygons'/f'{slug}_catchment_polygon_CANDIDATE.geojson'
        bbox_file=output_dir/'bounding_boxes'/f'{slug}_bounding_box_CANDIDATE.geojson'
        profile=dict(driver='GTiff',height=inner.shape[0],width=inner.shape[1],count=1,
                     dtype='uint8',crs=src_crs,transform=inner_tf,compress='lzw')
        # Atomic replacement: incomplete temporary files cannot masquerade as final outputs.
        tmp_mask=mask_file.with_name(mask_file.stem+'_INCOMPLETE.tif')
        with rasterio.open(tmp_mask,'w',**profile) as dst:
            dst.write(inner,1);dst.set_band_description(1,'1 upstream of MERIT computational outlet')
        tmp_mask.replace(mask_file)
        props={'station':station,'basin':basin,'tributary':st.tributary,'status':status,
               'source':'MERIT Hydro v1.0.1 D8; public ArcGIS dashboard point',
               'source_lon':float(st.original_lon),'source_lat':float(st.original_lat),
               'outlet_lon':float(st.snapped_lon),'outlet_lat':float(st.snapped_lat),
               'outlet_distance_m':float(st.distance_m),
               'official_area_km2':ref if ref_ok else None,
               'derived_area_km2':round(float(area),3),
               'area_difference_pct':round(float(pct),2) if ref_ok else None,
               'merit_upa_at_outlet_km2':round(float(st.merit_upa_km2),3),
               'derived_vs_merit_upa_pct':round(float(merit_pct),2),
               'touches_edge':touches_edge,'selection_method':st.selection_method,
               'source_notes':st.reference_notes}
        poly_fc={'type':'FeatureCollection','features':[{'type':'Feature','properties':props,'geometry':mapping(geom)}]}
        bounds=[float(v) for v in geom.bounds]
        bbox_fc={'type':'FeatureCollection','features':[{'type':'Feature','properties':dict(props,bbox_wgs84=bounds),
                            'geometry':mapping(box(*bounds))}]}
        write_json_if_changed(poly_file,poly_fc)
        write_json_if_changed(bbox_file,bbox_fc)
        rec={**props,'mask_file':str(mask_file.relative_to(REPO_ROOT).as_posix()),
             'polygon_file':str(poly_file.relative_to(REPO_ROOT).as_posix()),
             'bbox_file':str(bbox_file.relative_to(REPO_ROOT).as_posix()),
             'mask_cells':int(watershed.sum())}
        summary.append(rec)
        combined_features.append({'type':'Feature','properties':props,'geometry':mapping(geom)})
        print(f'{station}: {area:.1f} km², MERIT error {merit_pct:.2f}%, status={status}',flush=True)
    except Exception as exc:
        trace_failures.append({'station':station,'basin':basin,'error':str(exc)})
        print('FAILED:',station,str(exc),flush=True)

results=pd.DataFrame(summary)
results.to_csv(WORK_DIR/'additional_33_catchment_QA_registry.csv',index=False)
pd.DataFrame(trace_failures,columns=['station','basin','error']).to_csv(WORK_DIR/'delineation_failures.csv',index=False)
write_json_if_changed(WORK_DIR/'additional_33_CANDIDATE_combined.geojson',
    {'type':'FeatureCollection','features':combined_features})
print('GENERATED',len(results),'candidate polygon/mask/bbox triplets of 33; unresolved outlets:',len(unresolved_df),
      '| delineation errors:',len(trace_failures))
if not results.empty:
    display(results[['station','basin','derived_area_km2','official_area_km2','area_difference_pct',
                     'derived_vs_merit_upa_pct','outlet_distance_m','touches_edge','status']])

PASS: D8 reverse-tracing synthetic test
Putupaula: 2616.8 km², MERIT error 0.01%, status=CANDIDATE_PENDING_VISUAL_QA
Ellagawa: 1399.0 km², MERIT error 0.01%, status=CANDIDATE_PENDING_VISUAL_QA
Rathnapura: 633.7 km², MERIT error 0.01%, status=CANDIDATE_PENDING_VISUAL_QA
Kalawellawa (Millakanda): 772.8 km², MERIT error 0.01%, status=CANDIDATE_PENDING_VISUAL_QA
Magura: 160.9 km², MERIT error 0.01%, status=CANDIDATE_PENDING_VISUAL_QA
Baddegama: 721.6 km², MERIT error 0.01%, status=CANDIDATE_PENDING_VISUAL_QA
Thawalama: 358.5 km², MERIT error 0.01%, status=CANDIDATE_PENDING_VISUAL_QA
Panadugama: 434.2 km², MERIT error 0.01%, status=CANDIDATE_PENDING_VISUAL_QA
Pitabeddara: 288.1 km², MERIT error 0.01%, status=CANDIDATE_PENDING_VISUAL_QA
Urawa: 76.0 km², MERIT error 0.01%, status=REVIEW_OFFICIAL_DRAINAGE_DIFFERENCE_GT20
Thalgahagoda: 834.1 km², MERIT error 0.01%, status=CANDIDATE_PENDING_VISUAL_QA
Moraketiya: 1581.6 km², MERIT error 0.01%, status=CANDIDATE_PENDING_VISUAL_QA
Thanamalwila: 697.

,station,basin,derived_area_km2,official_area_km2,area_difference_pct,derived_vs_merit_upa_pct,outlet_distance_m,touches_edge,status
0,Putupaula,Kalu Ganga,2616.834,2598.0,0.72,0.01,76.71,False,CANDIDATE_PENDING_VISUAL_QA
1,Ellagawa,Kalu Ganga,1399.048,1393.0,0.43,0.01,80.18,False,CANDIDATE_PENDING_VISUAL_QA
2,Rathnapura,Kalu Ganga,633.669,603.0,5.09,0.01,31.31,False,CANDIDATE_PENDING_VISUAL_QA
3,Kalawellawa (Millakanda),Kalu Ganga,772.846,780.0,0.92,0.01,57.59,False,CANDIDATE_PENDING_VISUAL_QA
4,Magura,Kalu Ganga,160.912,152.0,5.86,0.01,50.52,False,CANDIDATE_PENDING_VISUAL_QA
5,Baddegama,Gin Ganga,721.554,749.0,3.66,0.01,58.04,False,CANDIDATE_PENDING_VISUAL_QA
6,Thawalama,Gin Ganga,358.513,377.0,4.90,0.01,69.22,False,CANDIDATE_PENDING_VISUAL_QA
7,Panadugama,Nilwala Ganga,434.170,445.0,2.43,0.01,17.94,False,CANDIDATE_PENDING_VISUAL_QA
8,Pitabeddara,Nilwala Ganga,288.131,310.0,7.05,0.01,47.03,False,CANDIDATE_PENDING_VISUAL_QA
9,Urawa,Nilwala Ganga,75.987,59.0,28.79,0.01,54.57,False,REVIEW_OFFICIAL_DRAINAGE_DIFFERENCE_GT20


## G. Candidate polygons on a basin-wise QA map

We use small basin-group maps rather than drawing all 33 detailed polygons at once (which would be slow in a browser). The full-resolution GeoJSON outputs remain unchanged; only the browser visualization uses simplified geometry. All source points and candidate outlets are displayed. We check overlapping/nested catchments, correct tributaries, potential confluence errors and hydrological divides. These files are saved only under `results/`.

In [39]:
PALETTE=['#fc8d59','#4dac26','#8073ac','#d7191c','#1b9e77','#e6ab02','#3182bd']
qa_map_paths=[]
if not results.empty:
    for basin,group in results.groupby('basin',sort=True):
        st_group=GAUGES[GAUGES.basin==basin]
        center=[float(st_group.lat.mean()),float(st_group.lon.mean())]
        qam=make_base_map(center,zoom=9,satellite_first=True,height=650)
        ee_tile_layer(MERIT.select('upa').gte(5).selfMask(),{'palette':['48c9ee']},
                      'MERIT drainage network ≥5 km²',True,.76).add_to(qam)
        ee_tile_layer(hyb_lines,{},'HydroBASINS L12 boundaries',False,.55).add_to(qam)
        for idx,rec in enumerate(group.itertuples(index=False)):
            source=REPO_ROOT/rec.polygon_file
            geom=shape(json.loads(source.read_text(encoding='utf8'))['features'][0]['geometry'])
            small=geom.simplify(.00045,preserve_topology=True)
            color=PALETTE[idx%len(PALETTE)]
            layer=folium.FeatureGroup(name=f'{rec.station}: {rec.status}',show=True)
            folium.GeoJson(mapping(small),style_function=lambda f,col=color:{'color':col,
                'weight':2.5,'fillColor':col,'fillOpacity':.13},
                tooltip=f'{rec.station}: candidate catchment {rec.derived_area_km2:.0f} km²').add_to(layer)
            folium.CircleMarker([rec.source_lat,rec.source_lon],radius=5,color='red',
                fill=True,tooltip=rec.station+' original point').add_to(layer)
            folium.CircleMarker([rec.outlet_lat,rec.outlet_lon],radius=4,color='yellow',
                fill=True,tooltip=rec.station+' selected computational outlet').add_to(layer)
            layer.add_to(qam)
        plugins.Fullscreen().add_to(qam);folium.LayerControl(collapsed=False).add_to(qam)
        path=save_browser_map(qam,WORK_DIR/f'{slugify(basin)}_candidate_catchment_QA.html',open_tab=False)
        qa_map_paths.append(path)
print('Created',len(qa_map_paths),'basin-level HTML QA maps.')
if qa_map_paths:
    print('Open desired maps via File Explorer in Chrome/Edge; do not display() Folium in VS Code.')

Map saved: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\aththanagalu_oya_candidate_catchment_QA.html
Map saved: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\deduru_oya_candidate_catchment_QA.html
Map saved: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\gin_ganga_candidate_catchment_QA.html
Map saved: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\heda_oya_candidate_catchment_QA.html
Map saved: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\kala_oya_candidate_catchment_QA.html
Map saved: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\kalu_ganga_candidate_catchment_QA.html
Map saved: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\kirindi_oya_candidate_catchment_QA.html
Map 

# FloodRisk Sri Lanka — Catchment QA Map Viewer (40 stations)
**Project:** J26-DS-314 · **Purpose:** inspect existing HTML maps and the combined 40-gauge catchment polygons.

We open the 18 **already generated** basin QA maps individually and create one additional combined preview from existing `shared/spatial/catchments/` GeoJSONs. No Earth Engine calls, MERIT downloads, watershed tracing, mask regeneration, or changes to the scientific outputs occur in this notebook.

**Prerequisites:** Run Sections F–H of the additional-33 notebook and the Kelani Section I export once. The 33 basin folders and the `kelani/polygons/` folder must exist in `shared/spatial/catchments/`.

**Important:** All geometries remain *CANDIDATE*. Individual gauge upstream catchments may overlap or nest naturally (especially along the same river). Uncovered land between gauged catchments does **not** indicate missing data or a failed delineation; these are not intended as a complete nationwide watershed partition.

## Setup — local file paths and browser helper

In [2]:
from pathlib import Path
import webbrowser

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

WORK_DIR = (
    REPO_ROOT / "results" / "catchments"
    / "additional_33_merit_hybas12"
)

def open_existing_basin_map(basin_label, basin_slug):
    file = WORK_DIR / f"{basin_slug}_candidate_catchment_QA.html"

    if not file.is_file():
        raise FileNotFoundError(f"Map not found: {file}")

    print(f"Opening {basin_label}: {file}")

    if not webbrowser.open_new_tab(file.resolve().as_uri()):
        print("Open manually:", file)

print("Original basin QA maps:",
      len(list(WORK_DIR.glob("*_candidate_catchment_QA.html"))))
print("Viewer ready.")

Original basin QA maps: 18
Viewer ready.


### 01. Aththanagalu Oya

In [32]:
# 01 — Aththanagalu Oya
open_existing_basin_map('Aththanagalu Oya', 'aththanagalu_oya')

Aththanagalu Oya QA map: c:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\aththanagalu_oya_candidate_catchment_QA.html


### 02. Deduru Oya

In [58]:
# 02 — Deduru Oya
open_existing_basin_map('Deduru Oya', 'deduru_oya')

Opening Deduru Oya: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\deduru_oya_candidate_catchment_QA.html


### 03. Gin Ganga

In [16]:
# 03 — Gin Ganga
open_existing_basin_map('Gin Ganga', 'gin_ganga')

Gin Ganga QA map: c:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\gin_ganga_candidate_catchment_QA.html


### 04. Heda Oya

In [17]:
# 04 — Heda Oya
open_existing_basin_map('Heda Oya', 'heda_oya')

Heda Oya QA map: c:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\heda_oya_candidate_catchment_QA.html


### 05. Kala Oya

In [19]:
# 05 — Kala Oya
open_existing_basin_map('Kala Oya', 'kala_oya')

Kala Oya QA map: c:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\kala_oya_candidate_catchment_QA.html


### 06. Kalu Ganga

In [3]:
# 06 — Kalu Ganga
open_existing_basin_map('Kalu Ganga', 'kalu_ganga')

Opening Kalu Ganga: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\kalu_ganga_candidate_catchment_QA.html


### 07. Kirindi Oya

In [20]:
# 07 — Kirindi Oya
open_existing_basin_map('Kirindi Oya', 'kirindi_oya')

Kirindi Oya QA map: c:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\kirindi_oya_candidate_catchment_QA.html


### 08. Kumbukkan Oya

In [21]:
# 08 — Kumbukkan Oya
open_existing_basin_map('Kumbukkan Oya', 'kumbukkan_oya')

Kumbukkan Oya QA map: c:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\kumbukkan_oya_candidate_catchment_QA.html


### 09. Ma Oya

In [57]:
# 09 — Ma Oya
open_existing_basin_map('Ma Oya', 'ma_oya')

Opening Ma Oya: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\ma_oya_candidate_catchment_QA.html


### 10. Maduru Oya

In [23]:
# 10 — Maduru Oya
open_existing_basin_map('Maduru Oya', 'maduru_oya')

Maduru Oya QA map: c:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\maduru_oya_candidate_catchment_QA.html


### 11. Maha Oya

In [ ]:
# 11 — Maha Oya
open_existing_basin_map('Maha Oya', 'maha_oya')

### 12. Mahaweli Ganga

In [24]:
# 12 — Mahaweli Ganga
open_existing_basin_map('Mahaweli Ganga', 'mahaweli_ganga')

Mahaweli Ganga QA map: c:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\mahaweli_ganga_candidate_catchment_QA.html


### 13. Malwathu Oya

In [ ]:
# 13 — Malwathu Oya
open_existing_basin_map('Malwathu Oya', 'malwathu_oya')

### 14. Mee Oya

In [ ]:
# 14 — Mee Oya
open_existing_basin_map('Mee Oya', 'mee_oya')

### 15. Menik Ganga

In [ ]:
# 15 — Menik Ganga
open_existing_basin_map('Menik Ganga', 'menik_ganga')

### 16. Nilwala Ganga

In [4]:
# 16 — Nilwala Ganga
open_existing_basin_map('Nilwala Ganga', 'nilwala_ganga')

Opening Nilwala Ganga: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\nilwala_ganga_candidate_catchment_QA.html


### 17. Walawe Ganga

In [ ]:
# 17 — Walawe Ganga
open_existing_basin_map('Walawe Ganga', 'walawe_ganga')

### 18. Yan Oya

In [56]:
# 18 — Yan Oya
open_existing_basin_map('Yan Oya', 'yan_oya')

Opening Yan Oya: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\yan_oya_candidate_catchment_QA.html


In [52]:
assert len(GAUGES) == 33
assert len(outlets_df) == 33
assert len(results) == 33

assert set(GAUGES.station) == set(results.station)
assert not results["touches_edge"].any()

assert (
    results["derived_vs_merit_upa_pct"] < 1
).all()

for column in ["polygon_file", "mask_file", "bbox_file"]:
    for relative_path in results[column]:
        assert (REPO_ROOT / relative_path).is_file(), relative_path

print("PASS: All 33 candidate catchments are ready for Section H.")

PASS: All 33 candidate catchments are ready for Section H.


## H. HydroBASINS cross-check, shared metadata, inventory and team ZIP

We record the HydroBASINS polygon intersecting each selected computational outlet, including its `HYBAS_ID`, `SUB_AREA`, `UP_AREA` and `NEXT_DOWN`. `UP_AREA` is the HydroBASINS sub-basin outlet's area **not an exact reference for the gauge**. This supports structural interpretation, not automatic accuracy certification.

The final table, checksums and README are written to `shared/spatial/catchments/` (and to appropriate per-basin metadata folders) **without a separate copy step**. A deterministic ZIP under `results/` packages the candidate spatial outputs for teammates. Rerunning this section recreates the ZIP under the same name and does not duplicate entries or touch Kelani products.

The final ZIP includes **only successfully generated candidates**. Missing-outlet and tracing failures are recorded under `results/catchments/additional_33_merit_hybas12/` and must be resolved before claiming 33/33 completed.

In [54]:
hyb_rows=[]
for st in outlets_df.itertuples(index=False):
    pt=ee.Geometry.Point([float(st.snapped_lon),float(st.snapped_lat)])
    subset=HYB.filterBounds(pt)
    n=int(subset.size().getInfo())
    props=(ee.Feature(subset.first()).toDictionary(['HYBAS_ID','SUB_AREA','UP_AREA','NEXT_DOWN']).getInfo()
           if n else {})
    hyb_rows.append({'station':st.station,'basin':st.basin,'matched_hybas12_polygons':n,
        'hybas12_id':props.get('HYBAS_ID'),'hybas12_SUB_AREA_km2':props.get('SUB_AREA'),
        'hybas12_UP_AREA_km2_NOT_exact_gauge_area':props.get('UP_AREA'),
        'hybas12_NEXT_DOWN':props.get('NEXT_DOWN'),
        'status':'INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED' if n==1 else 'REVIEW_MULTIPLE_OR_NO_SUBBASIN'})
hyb_qc=pd.DataFrame(hyb_rows)
hyb_qc.to_csv(WORK_DIR/'additional_33_hydrobasins_outlet_crosscheck.csv',index=False)
display(hyb_qc)

# Write one authoritative QA registry/metadata set for the additional 33 only.
AGG_PREFIX='additional_33'
registry_file=SHARED_ROOT/f'{AGG_PREFIX}_catchment_QA_registry.csv'
results.to_csv(registry_file,index=False)
for basin in GAUGES.basin.unique():
    p=SHARED_ROOT/slugify(basin)
    if not p.exists():continue
    metadata=p/'metadata';metadata.mkdir(exist_ok=True)
    if not results.empty:
        results[results.basin==basin].to_csv(metadata/'catchment_QA_registry.csv',index=False)
    if not hyb_qc.empty:
        hyb_qc[hyb_qc.basin==basin].to_csv(metadata/'hydrobasins_crosscheck.csv',index=False)
    if not outlets_df.empty:
        outlets_df[outlets_df.basin==basin].to_csv(metadata/'computational_outlet_candidates.csv',index=False)
    (p/'README.md').write_text(
      f'# {basin} — MERIT Hydro candidate catchments\n\n'
      'Source stations: Sri Lanka Irrigation Department dashboard GeoJSON geometry.\n\n'
      'MERIT Hydro v1.0.1 ESRI D8 flow directions, native raster; HydroBASINS L12 structural cross-check.\n\n'
      '**Status: CANDIDATE — not independently or officially certified.**\n'
      'Review exact tributary identity, outlet snapping, confluences, coastal/regulated rivers and area differences.\n'
      'Refer to metadata/catchment_QA_registry.csv and shared root manifests before modelling.\n',encoding='utf8')

# Include only files from this run; avoid Kelani and temporary cache files.
import hashlib
manifest=[]
paths=[]
for rec in results.itertuples(index=False):
    for kind in ['polygon_file','mask_file','bbox_file']:
        p=REPO_ROOT/getattr(rec,kind)
        if not p.exists():raise RuntimeError('Missing result: '+str(p))
        paths.append(p)
        manifest.append({'station':rec.station,'basin':rec.basin,'type':kind,
            'relative_path':str(p.relative_to(REPO_ROOT).as_posix()),
            'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),
            'qa_status':rec.status})
manifest_file=SHARED_ROOT/'additional_33_catchment_file_manifest.csv'
pd.DataFrame(manifest,columns=['station','basin','type','relative_path','bytes','sha256','qa_status']).to_csv(
    manifest_file,index=False)

zip_path=WORK_DIR/'additional_33_catchments_CANDIDATE_team_share.zip'
additional=[registry_file,manifest_file]
qa_readme = SHARED_ROOT / "QA_README.md"
if qa_readme.is_file():
    additional.append(qa_readme)
for basin in GAUGES.basin.unique():
    folder=SHARED_ROOT/slugify(basin)
    if folder.exists():
        additional += [p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in {'.csv','.md'}]
all_files={str(p.resolve()):p for p in paths+additional}
with ZipFile(zip_path,'w',ZIP_DEFLATED) as archive:
    for p in sorted(all_files.values(),key=lambda f:str(f)):
        archive.write(p,arcname=p.relative_to(REPO_ROOT).as_posix())
with ZipFile(zip_path) as archive:
    assert archive.testzip() is None
    assert len(archive.namelist())==len(set(archive.namelist()))
print('Generated:',len(results),'of 33 candidate sets; GeoJSON/mask/bbox files:',len(paths))
print('ZIP:',zip_path)
print('QA registry:',registry_file)
print('Unresolved/outlet failures:',len(unresolved_df),'delineation failures:',len(trace_failures))
print('No existing Kelani files were written or packaged.')

,station,basin,matched_hybas12_polygons,hybas12_id,hybas12_SUB_AREA_km2,hybas12_UP_AREA_km2_NOT_exact_gauge_area,hybas12_NEXT_DOWN,status
0,Putupaula,Kalu Ganga,1,4121159120,158.4,2753.4,4120049010,INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED
1,Ellagawa,Kalu Ganga,1,4121158870,135.5,1549.3,4121159030,INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED
2,Rathnapura,Kalu Ganga,1,4121158960,122.1,293.2,4121158930,INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED
3,Kalawellawa (Millakanda),Kalu Ganga,1,4121159050,226.1,914.7,4121159120,INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED
4,Magura,Kalu Ganga,1,4121159170,210.9,210.9,4121159050,INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED
5,Baddegama,Gin Ganga,1,4121602610,143.7,821.9,4120048950,INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED
6,Thawalama,Gin Ganga,1,4121602450,121.9,403.0,4121602540,INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED
7,Panadugama,Nilwala Ganga,1,4121160080,144.1,603.5,4121160220,INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED
8,Pitabeddara,Nilwala Ganga,1,4121602630,171.7,459.4,4121160080,INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED
9,Urawa,Nilwala Ganga,1,4121159900,151.9,151.9,4121602630,INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED


Generated: 33 of 33 candidate sets; GeoJSON/mask/bbox files: 99
ZIP: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\additional_33_merit_hybas12\additional_33_catchments_CANDIDATE_team_share.zip
QA registry: C:\Users\ASUS\projects\rp_project\J26-DS-314\shared\spatial\catchments\additional_33_catchment_QA_registry.csv
Unresolved/outlet failures: 0 delineation failures: 0
No existing Kelani files were written or packaged.


In [59]:
import json
from pathlib import Path
from shapely.geometry import Point, shape

repo = Path.cwd().resolve()
if repo.name.lower() == 'notebooks':
    repo = repo.parent

checks = [
    ('Urawa', 'nilwala_ganga', 'urawa', 80.57250000, 6.236666667),
    ('Horowpothana', 'yan_oya', 'horowpothana', 80.88083333, 8.576666667),
    ('Yaka Wewa', 'ma_oya', 'yaka_wewa', 80.67666667, 8.723333333),
    ('Moragaswewa', 'deduru_oya', 'moragaswewa', 79.99666667, 7.700000000),
    ('Kithulgala', 'kelani', 'kithulgala', 80.41916667, 6.991666667),
]

for name, basin, slug, lon, lat in checks:
    f = (repo / 'shared' / 'spatial' / 'catchments' / basin /
         'polygons' / f'{slug}_catchment_polygon_CANDIDATE.geojson')
    doc = json.loads(f.read_text(encoding='utf-8'))
    geom = shape(doc['features'][0]['geometry'])
    covered = geom.covers(Point(lon, lat))
    print(f'{name:16s} valid_polygon={geom.is_valid} outlet_in_polygon={covered}')
    if not geom.is_valid or not covered:
        print(f'  REVIEW SOURCE POLYGON: {f}')

Urawa            valid_polygon=True outlet_in_polygon=True
Horowpothana     valid_polygon=True outlet_in_polygon=True
Yaka Wewa        valid_polygon=True outlet_in_polygon=True
Moragaswewa      valid_polygon=True outlet_in_polygon=True
Kithulgala       valid_polygon=True outlet_in_polygon=True


## LOCAL-ONLY QA CHECK

In [60]:
from pathlib import Path
import json
from shapely.geometry import shape, Point

_here = Path.cwd().resolve()
_repo = next((p for p in (_here, *_here.parents)
              if (p / 'shared' / 'spatial' / 'catchments').is_dir()), None)
if _repo is None:
    raise FileNotFoundError('Open your notebook from inside the J26-DS-314 repository.')
_root = _repo / 'shared' / 'spatial' / 'catchments'
_files = sorted(_root.glob('*/polygons/*_catchment_polygon_CANDIDATE.geojson'))
assert len(_files) == 40, f'Expected exactly 40 station polygons; found {len(_files)}'

_rows = []
_seen = set()
for _path in _files:
    _fc = json.loads(_path.read_text(encoding='utf-8'))
    _features = _fc.get('features', [])
    if len(_features) != 1:
        raise ValueError(f'Expected one polygon feature: {_path}')
    _ft = _features[0]
    _props = _ft.get('properties', {})
    _name = _props.get('station', '')
    if not _name or _name in _seen:
        raise ValueError(f'Missing or duplicate station name: {_name!r} in {_path}')
    _seen.add(_name)
    _geom = shape(_ft['geometry'])
    _valid = (not _geom.is_empty and _geom.is_valid
              and _geom.geom_type in ('Polygon', 'MultiPolygon'))
    _outlet = Point(float(_props['outlet_lon']), float(_props['outlet_lat']))
    _covered = bool(_geom.covers(_outlet)) if _valid else False

    # Ensure the matching bbox exists and encloses the complete saved polygon.
    _stem = _path.name.removesuffix('_catchment_polygon_CANDIDATE.geojson')
    _bbox_path = (_path.parent.parent / 'bounding_boxes' /
                  f'{_stem}_bounding_box_CANDIDATE.geojson')
    _bbox_ok = False
    if _bbox_path.is_file() and _valid:
        _bf = json.loads(_bbox_path.read_text(encoding='utf-8'))['features']
        _bbox = shape(_bf[0]['geometry'])
        _bbox_ok = _bbox.is_valid and _bbox.buffer(1e-10).covers(_geom)

    _rows.append((_name, _valid, _covered, _bbox_ok))
    if not (_valid and _covered and _bbox_ok):
        print(f'REVIEW: {_name} | polygon_valid={_valid} | '
              f'outlet_covered={_covered} | bbox_covers_polygon={_bbox_ok}')

_ok = sum(a and b and c for _, a, b, c in _rows)
print(f'Complete: {_ok}/40 saved candidate polygons passed validity, outlet containment and bbox checks.')
if _ok != 40:
    raise AssertionError('One or more exported geometries need attention; see REVIEW lines above.')
print('PASS: 40/40 geometry checks. This is technical consistency, NOT boundary accuracy.')


Complete: 40/40 saved candidate polygons passed validity, outlet containment and bbox checks.
PASS: 40/40 geometry checks. This is technical consistency, NOT boundary accuracy.
